# Hindi Rhythm Scorer (Full Rules R1–R5)

Self-contained notebook combining the **full rule-based Hindi word scanner (rules R1–R4, plus optional R5)** with the **DP-based rhythm scorer** for Hindi verse. No local `.py` imports and no ML models are used.

### Part 1 — Word $\to$ Syllable Weights (`1` = laghu, `2` = guru)
- **Base rules:** Short vowels (`अ इ उ ऋ ऍ`, marks `ि ु ृ ॅ`) = `1`; long vowels (`आ ई ऊ ए ऐ ओ औ ॠ ऑ`, marks `ा ी ू े ै ो ौ ॄ`) = `2`; *anusvāra* (`ं`), *candrabindu* (`ँ`), *visarga* (`ः`) = `2`; a short syllable followed by a conjunct becomes `2`, except `म्ह`/`न्ह`/`ल्ह`.
- **R1:** `ॉ` (*candra O*) is **long** (`2`), moved out of short marks. `ॅ` (*candra E*) stays short (`1`, open question).
- **R2:** Normalize ASCII `:` to Devanagari visarga `ः` before scanning.
- **R3:** The `म्ह`/`न्ह`/`ल्ह` exception applies **only** when the cluster has exactly one consonant before `ह` (`lead_conjunct == 1`).
- **R4:** Compound & prefix splitting with weights reset at boundaries: parts come from leave-one-out merged gold (`>= 2` syllables, no single-letter words like `क`, `म`, `स`) or `PREFIX_LIST`, preferring fewest parts then longest first part. Controlled by `APPROVED_SPLITS` and `USE_ONLY_APPROVED = True` (applies approved splits + prefix list only) alongside `compound_candidates.tsv`.
- **R5 (optional):** Right-to-left bracket grouping where only a *bare-a* light joins its left neighbor.

### Part 2 — Rhythm Scoring (Flat Stream + DP)
- Legal feet sums: `2, 3, 4, 6`. Even feet (`2, 4, 6`) stand alone; odd (`3`) must be followed immediately by another odd (`3`) partner (borrowing across word boundaries allowed, never stranding a lone laghu).
- An unpaired odd group at the very end of the line is exempt.
- Failed odd lookahead $\to$ `1` violation: discard **only** that failed odd group (its word, or leftover slice if started mid-word) and restart at the very next word — words inspected during the failed lookahead are **not** skipped or consumed.
- Words whose total weight is not in `{2, 3, 4, 6}` are split by DP minimizing violations, then borrows.
- `line_score = clamp(1 - violations / number_of_words, 0, 1)`; `poem_score = mean(line_scores)`.

In [1]:
# Part 1: Data Loader, Hygiene Repairs, and Scanner (Rules R1-R4 + optional R5)
import csv
import json
import os
import re
from functools import lru_cache

# ---------------------------------------------------------------------------
# Data Hygiene & Gold Dataset Loader
# ---------------------------------------------------------------------------
ORIG_CORPUS_PATH = "Pronunciation_Consonants_Amisha.txt"
CORR_CORPUS_PATH = "Corrected_scansion_output_oneline_Consonants.txt"

BROKEN_JSON_REPAIRS = [
    ('["कद्","  दा ,"वर"]', '["कद्","दा","वर"]', 'कद्दावर: fixed unclosed quotes / extra space'),
    ('["कार्य,"पा","लि","का"]', '["कार्य","पा","लि","का"]', 'कार्यपालिका: fixed missing closing quote on कार्य'),
    ('["कू",जन"]', '["कू","जन"]', 'कूजन: fixed missing opening quote on जन'),
]

def load_and_clean_gold(orig_path=ORIG_CORPUS_PATH, corr_path=CORR_CORPUS_PATH):
    with open(orig_path, "r", encoding="utf-8") as f:
        orig_data = json.load(f)

    with open(corr_path, "r", encoding="utf-8") as f:
        corr_raw = f.read()

    repair_logs = []
    for old_str, new_str, desc in BROKEN_JSON_REPAIRS:
        if old_str in corr_raw:
            corr_raw = corr_raw.replace(old_str, new_str)
            repair_logs.append(desc)

    corr_data = json.loads(corr_raw)

    # Strip whitespace inside chunks (logged; `word` chunk field is ignored for scansion)
    ws_stripped = 0
    for d in (orig_data, corr_data):
        for v in d.values():
            chunks = v.get("word", [])
            cleaned = [c.strip() for c in chunks]
            if cleaned != chunks:
                ws_stripped += 1
            v["word"] = cleaned
    repair_logs.append(f"Stripped whitespace inside chunks in {ws_stripped} entries (chunk field ignored for scansion)")

    # Repair करकराहट [1.1] -> [1, 1]
    if "करकराहट" in corr_data:
        new_sc = []
        fixed_float = False
        for g in corr_data["करकराहट"]["scansion"]:
            new_g = []
            for x in g:
                if x == 1.1:
                    new_g.extend([1, 1])
                    fixed_float = True
                else:
                    new_g.append(int(x))
            new_sc.append(new_g)
        corr_data["करकराहट"]["scansion"] = new_sc
        if fixed_float:
            repair_logs.append("करकराहट: replaced weight typo [1.1] with [1, 1]")

    # Build merged gold: corrected file for क/क़ words, original for the rest
    # Ignore `word` chunks and store flat integer weights from `scansion` + source flag.
    merged_gold = {}
    for k, v in orig_data.items():
        if not (k.startswith("क") or k.startswith("क़")):
            merged_gold[k] = {
                "weights": [int(x) for g in v["scansion"] for x in g],
                "scansion": v["scansion"],
                "source": "uncorrected",
            }
    for k, v in corr_data.items():
        if k.startswith("क") or k.startswith("क़"):
            merged_gold[k] = {
                "weights": [int(x) for g in v["scansion"] for x in g],
                "scansion": v["scansion"],
                "source": "sir_corrected",
            }

    return merged_gold, repair_logs

merged_gold, hygiene_logs = load_and_clean_gold()
print("Data hygiene applied:")
for msg in hygiene_logs:
    print("  -", msg)
print(
    f"Merged gold loaded: {len(merged_gold)} words "
    f"(corrected क/क़: {sum(1 for v in merged_gold.values() if v['source'] == 'sir_corrected')}, "
    f"uncorrected rest: {sum(1 for v in merged_gold.values() if v['source'] == 'uncorrected')})"
)

# ---------------------------------------------------------------------------
# Phonology & Syllabifier (Base + R1, R2, R3)
# ---------------------------------------------------------------------------
# Open question for Sir: ॅ (candra E) remains short pending corpus evidence.
SHORT_MATRAS_BASE = {'ि', 'ु', 'ृ', 'ॅ', 'ॉ'}
LONG_MATRAS_BASE  = {'ा', 'ी', 'ू', 'े', 'ै', 'ो', 'ौ', 'ॄ'}
SHORT_MATRAS_R1   = {'ि', 'ु', 'ृ', 'ॅ'}
LONG_MATRAS_R1    = {'ा', 'ी', 'ू', 'े', 'ै', 'ो', 'ौ', 'ॄ', 'ॉ'}

SHORT_INDEP_VOWELS = {'अ', 'इ', 'उ', 'ऋ', 'ऍ'}
LONG_INDEP_VOWELS  = {'आ', 'ई', 'ऊ', 'ए', 'ऐ', 'ओ', 'औ', 'ॠ', 'ऑ'}

ANUSVARA = 'ं'
CHANDRABINDU = 'ँ'
VISARGA = 'ः'
VIRAMA = '्'
NUKTA = '़'

HA = 'ह'
NASAL_LATERAL_ONSETS = {'म', 'न', 'ल'}

def is_consonant(ch):
    o = ord(ch)
    return (0x0915 <= o <= 0x0939) or (0x0958 <= o <= 0x095F)

def normalize_word(word, apply_r2=True):
    """R2: Normalize ASCII ':' to Devanagari visarga 'ः'."""
    return word.replace(':', VISARGA) if apply_r2 else word

def syllabify(word, apply_r1=True, apply_r2=True):
    word = normalize_word(word, apply_r2=apply_r2)
    short_m = SHORT_MATRAS_R1 if apply_r1 else SHORT_MATRAS_BASE
    long_m = LONG_MATRAS_R1 if apply_r1 else LONG_MATRAS_BASE
    vowel_signs = short_m | long_m
    indep_vowels = SHORT_INDEP_VOWELS | LONG_INDEP_VOWELS

    i, n = 0, len(word)
    syllables = []
    while i < n:
        ch = word[i]
        if ch in indep_vowels:
            start = i
            long_ = ch in LONG_INDEP_VOWELS
            i += 1
            nasal = False
            if i < n and word[i] in (ANUSVARA, CHANDRABINDU, VISARGA):
                nasal = True
                i += 1
            syllables.append({
                'text': word[start:i],
                'long': long_,
                'nasal': nasal,
                'lead_conjunct': 0,
                'has_own_vowel': True,
                'final_cons': None,
                'onset_first': None,
            })
            continue

        if is_consonant(ch):
            lead_conjunct = 0
            start = i
            dangling = False
            final_cons = None
            onset_first = ch
            cur_base = ch

            while True:
                j = i + 1
                if j < n and word[j] == NUKTA:
                    j += 1
                if j < n and word[j] == VIRAMA:
                    i = j + 1
                    lead_conjunct += 1
                    if i < n and is_consonant(word[i]):
                        cur_base = word[i]
                        continue
                    else:
                        dangling = True
                        break
                else:
                    final_cons = cur_base
                    i = j
                    break

            long_ = False
            nasal = False
            has_own_vowel = not dangling

            if not dangling and i < n and word[i] in vowel_signs:
                long_ = word[i] in long_m
                i += 1

            if not dangling and i < n and word[i] in (ANUSVARA, CHANDRABINDU, VISARGA):
                nasal = True
                i += 1

            syllables.append({
                'text': word[start:i],
                'long': long_,
                'nasal': nasal,
                'lead_conjunct': lead_conjunct,
                'has_own_vowel': has_own_vowel,
                'final_cons': final_cons,
                'onset_first': onset_first if lead_conjunct >= 1 else None,
            })
            continue

        i += 1
    return syllables

def assign_part_weights(part, is_final=True, apply_r1=True, apply_r2=True, apply_r3=True):
    """Assign laghu/guru weights to one word or compound part."""
    syls = syllabify(part, apply_r1=apply_r1, apply_r2=apply_r2)
    if not syls:
        return [], syls
    weights = [
        1 if not s['has_own_vowel'] else (2 if (s['long'] or s['nasal']) else 1)
        for s in syls
    ]
    for idx in range(1, len(syls)):
        s = syls[idx]
        if apply_r3:
            # R3: exception holds ONLY when cluster has exactly one consonant + ह
            exc = (s['lead_conjunct'] == 1 and s['final_cons'] == HA and s['onset_first'] in NASAL_LATERAL_ONSETS)
        else:
            exc = (s['final_cons'] == HA and s['onset_first'] in NASAL_LATERAL_ONSETS)
        if s['lead_conjunct'] >= 1 and s['has_own_vowel'] and not exc:
            weights[idx - 1] = 2

    # A non-final virama prefix (e.g. दुर्-, निर्-, उत्-, सम्-) ends in a coda consonant
    # closing its preceding syllable rather than emitting an independent laghu syllable.
    if not is_final and not syls[-1]['has_own_vowel'] and len(weights) >= 2:
        weights[-2] = 2
        weights.pop()
        syls.pop()
    return weights, syls

# ---------------------------------------------------------------------------
# R4: Compound & Prefix Splitting
# ---------------------------------------------------------------------------
PREFIX_LIST = [
    'सु', 'कु', 'वि', 'प्र', 'नि', 'दु', 'अति', 'अधि', 'अनु', 'अभि',
    'उप', 'परा', 'परि', 'महा', 'अप', 'अव', 'उत्', 'निर्', 'दुर्',
    'प्रति', 'सम्', 'अन्', 'अ'
]
PREFIXES_BY_LEN = sorted(PREFIX_LIST, key=len, reverse=True)

# Leave-one-out dictionary parts: must have >= 2 syllables (excludes single-letter क, म, स, etc.)
DICT_VOCAB_GE2 = {
    k for k in merged_gold
    if len(syllabify(k, apply_r1=True, apply_r2=True)) >= 2 and not k.endswith(VIRAMA)
}

def discover_split(word, vocab=DICT_VOCAB_GE2, prefix_list=PREFIX_LIST):
    """
    Find best split (fewest parts, then longest first part) using leave-one-out dict (>=2 syls)
    and prefix list. Returns (parts_list, source) or (None, None).
    """
    n = len(word)
    # 1. Try 2-part splits (fewest parts); longest first part wins
    for i in range(n - 1, 0, -1):
        p1, p2 = word[:i], word[i:]
        if p2 in vocab and p2 != word:
            if p1 in prefix_list:
                return [p1, p2], "prefix"
            if p1 in vocab and p1 != word:
                return [p1, p2], "dictionary"

    # 2. Try 3-part splits if no 2-part split exists
    for i in range(n - 2, 0, -1):
        p1, rem = word[:i], word[i:]
        is_pref = p1 in prefix_list
        is_dict = (p1 in vocab and p1 != word)
        if not (is_pref or is_dict):
            continue
        for j in range(len(rem) - 1, 1, -1):
            p2, p3 = rem[:j], rem[j:]
            if p2 in vocab and p3 in vocab and p2 != word and p3 != word:
                src = "prefix+dictionary" if is_pref else "dictionary"
                return [p1, p2, p3], src

    return None, None

# Discover all compound candidates across merged gold and write compound_candidates.tsv
CANDIDATE_MAP = {}
tsv_rows = []
for w in sorted(merged_gold):
    parts, src = discover_split(w)
    if parts:
        CANDIDATE_MAP[w] = (parts, src)
        tsv_rows.append({"word": w, "parts": " + ".join(parts), "source": src})

with open("compound_candidates.tsv", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["word", "parts", "source"], delimiter="\t")
    writer.writeheader()
    writer.writerows(tsv_rows)
print(f"Wrote compound_candidates.tsv with {len(tsv_rows)} candidate splits (columns: word, parts, source).")

# Editable set of approved dictionary compound splits (user/Sir can edit this set)
APPROVED_SPLITS = {
    'कार्यक्रम',
    'कालक्रम',
    'कालत्रय',
    'कार्यप्रणाली',
}
USE_ONLY_APPROVED = True

def get_r4_parts(word, use_only_approved=None):
    """Return compound/prefix parts according to USE_ONLY_APPROVED."""
    if use_only_approved is None:
        use_only_approved = USE_ONLY_APPROVED

    if word in APPROVED_SPLITS:
        if word in CANDIDATE_MAP:
            return CANDIDATE_MAP[word][0], "approved"
        pts, _ = discover_split(word)
        if pts:
            return pts, "approved"

    if use_only_approved:
        # Apply approved splits plus prefix list only (rest >= 2 syllables in gold dict, leave-one-out)
        for p in PREFIXES_BY_LEN:
            if word.startswith(p) and len(word) > len(p):
                rest = word[len(p):]
                if rest in DICT_VOCAB_GE2 and rest != word:
                    return [p, rest], "prefix"
        return None, None

    if word in CANDIDATE_MAP:
        return CANDIDATE_MAP[word]
    return discover_split(word)

def scan_word(word, apply_r1=True, apply_r2=True, apply_r3=True, apply_r4=True, use_only_approved=None):
    """Scan a word -> (syllables, flat_weights, split_parts, split_source)."""
    norm_w = normalize_word(word, apply_r2=apply_r2)
    if apply_r4:
        parts, src = get_r4_parts(norm_w, use_only_approved=use_only_approved)
        if parts:
            all_weights, all_syls = [], []
            for idx, part in enumerate(parts):
                ws, syls = assign_part_weights(
                    part, is_final=(idx == len(parts) - 1),
                    apply_r1=apply_r1, apply_r2=apply_r2, apply_r3=apply_r3
                )
                all_weights.extend(ws)
                all_syls.extend(syls)
            return all_syls, all_weights, parts, src

    ws, syls = assign_part_weights(norm_w, is_final=True, apply_r1=apply_r1, apply_r2=apply_r2, apply_r3=apply_r3)
    return syls, ws, None, None

# ---------------------------------------------------------------------------
# R5 (Optional): Bare-a Right-to-Left Bracket Grouping
# ---------------------------------------------------------------------------
def bare_a(s):
    """True if syllable is a light bare consonantal 'a' (no vowel mark, not an independent vowel)."""
    vowel_signs = SHORT_MATRAS_R1 | LONG_MATRAS_R1
    indep_vowels = SHORT_INDEP_VOWELS | LONG_INDEP_VOWELS
    return s['has_own_vowel'] and not any(c in vowel_signs or c in indep_vowels for c in s['text'])

def group_syllables(syl, ws):
    """Optional R5 bracket grouping (right-to-left: only bare-a light joins left neighbor)."""
    out, i = [], len(ws) - 1
    while i >= 0:
        if ws[i] == 1 and bare_a(syl[i]) and i > 0:
            out.append([ws[i - 1], ws[i]])
            i -= 2
        else:
            out.append([ws[i]])
            i -= 1
    return out[::-1]

Data hygiene applied:
  - कद्दावर: fixed unclosed quotes / extra space
  - कार्यपालिका: fixed missing closing quote on कार्य
  - कूजन: fixed missing opening quote on जन
  - Stripped whitespace inside chunks in 7 entries (chunk field ignored for scansion)
  - करकराहट: replaced weight typo [1.1] with [1, 1]
Merged gold loaded: 13225 words (corrected क/क़: 1056, uncorrected rest: 12169)
Wrote compound_candidates.tsv with 1102 candidate splits (columns: word, parts, source).


In [2]:
# Part 1 Validation Tables & Rule Assertions (Before vs After R1-R4)
corr_k = {k: v["weights"] for k, v in merged_gold.items() if v["source"] == "sir_corrected"}
rest   = {k: v["weights"] for k, v in merged_gold.items() if v["source"] == "uncorrected"}

CONFIGS = [
    ("Base (pre R1-R4)",                  False, False, False, False, True),
    ("+ R1 (ऑ/ॉ long)",                   True,  False, False, False, True),
    ("+ R1-R2 (: -> ः)",                  True,  True,  False, False, True),
    ("+ R1-R3 (म्ह/न्ह/ल्ह exact=1)",     True,  True,  True,  False, True),
    ("+ R1-R4 (all dict+prefix)",         True,  True,  True,  True,  False),
    ("+ R1-R4 (APPROVED + prefix only)",  True,  True,  True,  True,  True),
]

eval_preds = {}
print(f"{'Stage':34s} | {'Corrected k-section':22s} | {'Uncorrected rest':22s}")
print("-" * 85)
for label, r1, r2, r3, r4, app_only in CONFIGS:
    stage_preds = {}
    stats = {}
    for sub_name, sub in [("corr_k", corr_k), ("rest", rest)]:
        preds = {w: scan_word(w, r1, r2, r3, r4, app_only)[1] for w in sub}
        hits = sum(1 for w, g in sub.items() if preds[w] == g)
        stage_preds[sub_name] = preds
        stats[sub_name] = (hits, len(sub), 100.0 * hits / len(sub))
    eval_preds[label] = stage_preds
    ck, nk, pk = stats["corr_k"]
    cr, nr, pr = stats["rest"]
    print(f"{label:34s} | {ck:4d}/{nk:<4d} ({pk:6.2f}%)    | {cr:5d}/{nr:<5d} ({pr:6.2f}%)")

def print_delta(prev_label, curr_label, max_show=30):
    print(f"\n--- Delta: {prev_label}  ==>  {curr_label} ---")
    for sub_name, sub in [("corr_k", corr_k), ("rest", rest)]:
        p_prev = eval_preds[prev_label][sub_name]
        p_curr = eval_preds[curr_label][sub_name]
        fixed = sorted(w for w, g in sub.items() if p_prev[w] != g and p_curr[w] == g)
        broke = sorted(w for w, g in sub.items() if p_prev[w] == g and p_curr[w] != g)
        print(f"  [{sub_name}] Fixed ({len(fixed)}): {fixed[:max_show]}" + (f" ... (+{len(fixed)-max_show} more)" if len(fixed) > max_show else ""))
        print(f"  [{sub_name}] Broke ({len(broke)}): {broke[:max_show]}" + (f" ... (+{len(broke)-max_show} more)" if len(broke) > max_show else ""))

print_delta("Base (pre R1-R4)", "+ R1 (ऑ/ॉ long)")
print_delta("+ R1 (ऑ/ॉ long)", "+ R1-R2 (: -> ः)")
print_delta("+ R1-R2 (: -> ः)", "+ R1-R3 (म्ह/न्ह/ल्ह exact=1)")
print_delta("+ R1-R3 (म्ह/न्ह/ल्ह exact=1)", "+ R1-R4 (all dict+prefix)")
print_delta("+ R1-R3 (म्ह/न्ह/ल्ह exact=1)", "+ R1-R4 (APPROVED + prefix only)")
print_delta("Base (pre R1-R4)", "+ R1-R4 (APPROVED + prefix only)")

# Remaining mismatches on corr_k under final APPROVED + prefix only
final_preds_k = eval_preds["+ R1-R4 (APPROVED + prefix only)"]["corr_k"]
rem_k = [(w, final_preds_k[w], corr_k[w]) for w in sorted(corr_k) if final_preds_k[w] != corr_k[w]]
print(f"\nRemaining disagreements on corrected k-section ({len(rem_k)}):")
for w, pred, gold in rem_k:
    print(f"  {w:14s}: pred={pred}  gold={gold}")

# ---------------------------------------------------------------------------
# Rule Acceptance Assertions (R1-R4)
# ---------------------------------------------------------------------------
for w in ["कॉपी", "कॉलर", "कॉलेज"]:
    assert scan_word(w)[1] == corr_k[w], f"R1 failed on {w}"
assert scan_word("क्रमश:")[1] == corr_k["क्रमश:"] == scan_word("क्रमशः")[1], "R2 failed on क्रमश:"
assert scan_word("छी:छी:")[1] == rest["छी:छी:"], "R2 failed on छी:छी:"
for w in ["चिन्न्ह", "ब्रम्म्ह", "ब्रम्म्हा", "ब्रम्म्हाण", "ब्रम्म्ही"]:
    assert scan_word(w)[1] == rest[w], f"R3 failed on {w}"
for w in ["कार्यक्रम", "कालक्रम", "कालत्रय", "कार्यप्रणाली", "कुख्यात"]:
    assert scan_word(w)[1] == corr_k[w], f"R4 failed on {w}: got {scan_word(w)[1]} vs {corr_k[w]}"
assert get_r4_parts("कनस्तर")[0] is None and scan_word("कनस्तर")[1] == corr_k["कनस्तर"], "R4 failed: कनस्तर must NOT split"
print("\nPASS: All Part 1 (R1-R4) acceptance assertions passed!")

Stage                              | Corrected k-section    | Uncorrected rest      
-------------------------------------------------------------------------------------
Base (pre R1-R4)                   | 1037/1056 ( 98.20%)    | 12156/12169 ( 99.89%)
+ R1 (ऑ/ॉ long)                    | 1040/1056 ( 98.48%)    | 12141/12169 ( 99.77%)
+ R1-R2 (: -> ः)                   | 1041/1056 ( 98.58%)    | 12141/12169 ( 99.77%)
+ R1-R3 (म्ह/न्ह/ल्ह exact=1)      | 1041/1056 ( 98.58%)    | 12146/12169 ( 99.81%)
+ R1-R4 (all dict+prefix)          | 1044/1056 ( 98.86%)    | 12079/12169 ( 99.26%)
+ R1-R4 (APPROVED + prefix only)   | 1045/1056 ( 98.96%)    | 12102/12169 ( 99.45%)

--- Delta: Base (pre R1-R4)  ==>  + R1 (ऑ/ॉ long) ---
  [corr_k] Fixed (3): ['कॉपी', 'कॉलर', 'कॉलेज']
  [corr_k] Broke (0): []
  [rest] Fixed (0): []
  [rest] Broke (15): ['टाउनहॉल', 'डॉलर', 'डॉवाँडोल', 'प्लॉट', 'फॉल', 'फ्रॉक', 'बॉयकॉट', 'बॉयलर', 'ब्लॉक', 'मॉडल', 'रॉयल्टी', 'रॉोचना', 'लॉबी', 'स्टॉक', 'स्टॉप']

--- Delta: +

In [3]:
# Part 2: Rhythm Scorer (flat stream + DP; restart-at-next-word on unpaired odd)
from dataclasses import dataclass, field
from typing import Callable, List, Optional, Sequence

FEET = (2, 3, 4, 6)   # legal rhythmic groups: dwikal, trikal, chaukal, shatkal
PARTNER = 3           # an odd group must be followed by a trikal (sum exactly 3)

@dataclass
class Event:
    kind: str                 # 'even' | 'pair' | 'trailing_odd' | 'violation'
    words: List[int]          # indices of the words this event touches
    text: str                 # human-readable, e.g. "चलो | हम+कहीं[0:1]"
    borrowed: bool = False    # a word was cut to form the partner

@dataclass
class LineResult:
    words: List[str]
    score: float
    violations: int
    pairs: int
    borrows: int
    even_feet: int
    trailing_odd: int
    discarded_words: int
    irregular_words: int      # words whose total is not 2/3/4/6
    events: List[Event] = field(default_factory=list)

@dataclass
class PoemResult:
    score: float
    lines: List[LineResult]

def analyse_line(word_weights: Sequence[Sequence[int]], words: Optional[Sequence[str]] = None,
                 normalise: str = "words", severity: float = 1.0) -> LineResult:
    nw = len(word_weights)
    names = list(words) if words is not None else [str(i) for i in range(nw)]
    flat, word_of, w_start, w_end = [], [], [], []
    for k, ws in enumerate(word_weights):
        w_start.append(len(flat)); flat.extend(ws); word_of.extend([k] * len(ws)); w_end.append(len(flat))
    n = len(flat)
    pre = [0]
    for x in flat:
        pre.append(pre[-1] + x)
    S = lambda a, b: pre[b] - pre[a]

    def foot_options(p):
        k = word_of[p]; e = w_end[k]
        if p == w_start[k] and S(p, e) in FEET:
            return [(e, S(p, e))]                      # regular word: one whole foot
        opts = [(q, S(p, q)) for q in range(e, p, -1)
                if S(p, q) in FEET and S(q, e) != 1]   # never strand a lone laghu
        return opts or [(e, S(p, e))]                  # fallback: whole remainder, parity only

    def partner_from(q):
        """Shortest syllable run after q with sum exactly 3, or None if unavailable."""
        r, t = 0, q
        while t < n:
            r += flat[t]; t += 1
            if r == PARTNER:
                kw = word_of[t - 1]
                if S(t, w_end[kw]) == 1:               # cut would strand a lone laghu
                    return None
                return t
            if r > PARTNER:
                return None
        return None                                    # stream exhausted

    INF = (10 ** 9, 10 ** 9)
    best = [INF] * (n + 1); choice = [None] * (n + 1)
    best[n] = (0, 0)
    for p in range(n - 1, -1, -1):
        for q, s in foot_options(p):
            if s % 2 == 0:
                cand, nxt, info = best[q], q, ('even', p, q, None)
            elif q == n:
                cand, nxt, info = (0, 0), n, ('trailing_odd', p, q, None)
            else:
                m = partner_from(q)
                if m is not None:
                    b = 1 if m < w_end[word_of[m - 1]] else 0
                    cand = (best[m][0], best[m][1] + b); nxt = m; info = ('pair', p, q, m)
                else:
                    # Discard only the failed odd group (word or mid-word leftover); restart at next word
                    resume = w_end[word_of[p]]
                    cand = (best[resume][0] + 1, best[resume][1]); nxt = resume
                    info = ('violation', p, resume, None)
            if cand < best[p]:
                best[p] = cand; choice[p] = (nxt, info)

    def wspan(a, b):
        return sorted({word_of[i] for i in range(a, b)})

    def label(a, b):
        out = []
        for k in wspan(a, b):
            lo, hi = max(a, w_start[k]), min(b, w_end[k])
            piece = names[k] if (lo, hi) == (w_start[k], w_end[k]) else f"{names[k]}[{lo - w_start[k]}:{hi - w_start[k]}]"
            out.append(piece)
        return "+".join(out)

    events, p = [], 0
    while p < n:
        nxt, (kind, a, b, m) = choice[p]
        if kind == 'pair':
            events.append(Event('pair', wspan(a, m), f"{label(a, b)} | {label(b, m)}",
                                borrowed=m < w_end[word_of[m - 1]]))
        elif kind == 'violation':
            msg = f"unpaired odd {label(a, b)}; discarded"
            if b < n:
                msg += f"; restarting at {names[word_of[b]]}"
            events.append(Event('violation', wspan(a, b), msg))
        else:
            events.append(Event(kind, wspan(a, b), label(a, b)))
        p = nxt

    V = sum(e.kind == 'violation' for e in events)
    pairs = sum(e.kind == 'pair' for e in events)
    evens = sum(e.kind == 'even' for e in events)
    trail = sum(e.kind == 'trailing_odd' for e in events)
    discarded = sum(len(e.words) for e in events if e.kind == 'violation')
    irregular = sum(1 for ws in word_weights if sum(ws) not in FEET)

    if normalise == "words":
        denom = max(1, nw)
    elif normalise == "units":
        denom = max(1, evens + pairs + trail + V)
    else:
        raise ValueError(normalise)
    score = max(0.0, min(1.0, 1.0 - severity * V / denom))
    return LineResult(names, score, V, pairs, best[0][1] if n else 0, evens, trail, discarded, irregular, events)

_PUNCT = re.compile(r"[।॥,.;:!?\"'“”‘’()\[\]{}<>…‌‍*_/\\|]+")
_DEVANAGARI = re.compile(r"[ऀ-ॿ]")

def tokenize(line: str) -> List[str]:
    # R2: preserve ASCII ':' attached to Devanagari letters as visarga before stripping punctuation
    line = re.sub(r"(?<=[ऀ-ॿ]):", VISARGA, line)
    line = _PUNCT.sub(" ", line.replace("-", " ").replace("–", " ").replace("—", " "))
    return [t for t in line.split() if _DEVANAGARI.search(t)]

@lru_cache(maxsize=None)
def default_scan(word: str) -> tuple:
    """Flat laghu/guru weights from Part 1 full-rule scanner (R1-R4)."""
    return tuple(scan_word(word)[1])

def score_line(line: str, scan: Callable[[str], Sequence[int]] = default_scan, **kw) -> Optional[LineResult]:
    toks = [t for t in tokenize(line) if len(scan(t)) > 0]
    if not toks:
        return None
    return analyse_line([list(scan(t)) for t in toks], toks, **kw)

def score_poem(text: str, scan: Callable[[str], Sequence[int]] = default_scan, **kw) -> PoemResult:
    lines = [r for r in (score_line(l, scan, **kw) for l in text.splitlines()) if r is not None]
    return PoemResult(sum(r.score for r in lines) / len(lines) if lines else float("nan"), lines)

def split_poem_lines(poem_text: str) -> List[str]:
    """Split a pasted poem on Devanagari full stops (।, ॥), ASCII '.', or newlines."""
    return [seg.strip() for seg in re.findall(r"[^।॥.\r\n]+[।॥.]?", poem_text) if _DEVANAGARI.search(seg)]

def display_rhythm_report(lines: Sequence[str]) -> Optional[PoemResult]:
    """Print line-by-line weights, rhythm events, line scores, and overall poem score."""
    SYM = {'even': 'EVEN ', 'pair': 'PAIR ', 'trailing_odd': 'END  ', 'violation': 'FAIL '}
    scored = []
    for raw_line in lines:
        line = raw_line.strip()
        if not line:
            continue
        r = score_line(line)
        print("=" * 72)
        print("Line:", line)
        if r is None:
            print("  (no Devanagari words found)")
            continue

        word_strs = []
        for w in r.words:
            _, ws, parts, src = scan_word(w)
            split_tag = f" [{'+'.join(parts)}:{src}]" if parts else ""
            word_strs.append(f"{w}={ws}(sum {sum(ws)}{split_tag})")
        print("Words:", "  ".join(word_strs))

        print("Events:")
        for e in r.events:
            borrow_tag = "  [borrow]" if e.borrowed else ""
            print(f"   {SYM[e.kind]} {e.text}{borrow_tag}")

        print(
            f"Violations: {r.violations} | pairs: {r.pairs} | borrows: {r.borrows} | "
            f"even_feet: {r.even_feet} | trailing_odd: {r.trailing_odd} | words: {len(r.words)}"
        )
        print(f"LINE SCORE: {r.score:.3f}")
        scored.append(r)

    print("=" * 72)
    if scored:
        mean_score = sum(r.score for r in scored) / len(scored)
        print(f"POEM SCORE (mean of {len(scored)} lines): {mean_score:.3f}")
        return PoemResult(mean_score, scored)
    return None

# ---------------------------------------------------------------------------
# Rhythm Acceptance Tests & 20,000-line Fuzz Check
# ---------------------------------------------------------------------------
r_ex1 = score_line("चलो हम कहीं दूर चले")
assert r_ex1 is not None and r_ex1.violations == 0 and r_ex1.borrows == 1 and r_ex1.score == 1.0, r_ex1

r_ex2 = score_line("चलो हम और कहीं चले")
assert r_ex2 is not None and r_ex2.violations == 1, r_ex2
assert r_ex2.pairs == 1 and r_ex2.even_feet == 1 and r_ex2.trailing_odd == 1, r_ex2
assert r_ex2.discarded_words == 1 and abs(r_ex2.score - 0.8) < 1e-9, r_ex2
assert [e.text for e in r_ex2.events] == [
    "unpaired odd चलो; discarded; restarting at हम",
    "हम",
    "और | कहीं",
    "चले",
], r_ex2.events

# Mid-word leftover failure: Word 0 borrows w1[0:3]; leftover w1[3:5]=[2,1] (odd) fails and discards only leftover, restarting at w2
r_left = analyse_line([[1, 2], [1, 1, 1, 2, 1], [2]], ["w0", "w1", "w2"])
assert r_left.violations == 1 and r_left.pairs == 1 and r_left.even_feet == 1, r_left
assert [e.text for e in r_left.events] == [
    "w0 | w1[0:3]",
    "unpaired odd w1[3:5]; discarded; restarting at w2",
    "w2",
], r_left.events

assert analyse_line([[2, 2], [1, 2]]).violations == 0
assert analyse_line([[1, 2], [2]]).violations == 1
assert analyse_line([[2, 2, 1], [1, 2]]).violations == 0
assert analyse_line([[1, 2], [1, 2], [1, 2]]).pairs == 1

import random
rnd = random.Random(0)
for _ in range(20000):
    line = [[rnd.choice((1, 2)) for _ in range(rnd.randint(1, 6))] for _ in range(rnd.randint(1, 14))]
    for norm in ("words", "units"):
        a = analyse_line(line, normalise=norm)
        b = analyse_line(line, normalise=norm)
        assert 0.0 <= a.score <= 1.0 and a.violations <= len(line) and a.score == b.score

print("PASS: All Part 2 acceptance tests + 20,000-line fuzz x 2 normalisers passed!")

PASS: All Part 2 acceptance tests + 20,000-line fuzz x 2 normalisers passed!


In [4]:
# >>> EDIT HERE: one Hindi line per row <<<
LINES = """
चलो हम कहीं दूर चले
चलो हम और कहीं चले
कार्यक्रम बहुत सुगम है
"""

In [5]:
# Output Cell: Word Weights, Rhythm Pairs/Events, Line Scores, and Poem Score
display_rhythm_report([l for l in LINES.strip().splitlines() if l.strip()])

Line: चलो हम कहीं दूर चले
Words: चलो=[1, 2](sum 3)  हम=[1, 1](sum 2)  कहीं=[1, 2](sum 3)  दूर=[2, 1](sum 3)  चले=[1, 2](sum 3)
Events:
   PAIR  चलो | हम+कहीं[0:1]  [borrow]
   EVEN  कहीं[1:2]
   PAIR  दूर | चले
Violations: 0 | pairs: 2 | borrows: 1 | even_feet: 1 | trailing_odd: 0 | words: 5
LINE SCORE: 1.000
Line: चलो हम और कहीं चले
Words: चलो=[1, 2](sum 3)  हम=[1, 1](sum 2)  और=[2, 1](sum 3)  कहीं=[1, 2](sum 3)  चले=[1, 2](sum 3)
Events:
   FAIL  unpaired odd चलो; discarded; restarting at हम
   EVEN  हम
   PAIR  और | कहीं
   END   चले
Violations: 1 | pairs: 1 | borrows: 0 | even_feet: 1 | trailing_odd: 1 | words: 5
LINE SCORE: 0.800
Line: कार्यक्रम बहुत सुगम है
Words: कार्यक्रम=[2, 1, 1, 1](sum 5 [कार्य+क्रम:approved])  बहुत=[1, 1, 1](sum 3)  सुगम=[1, 1, 1](sum 3)  है=[2](sum 2)
Events:
   EVEN  कार्यक्रम[0:1]
   PAIR  कार्यक्रम[1:4] | बहुत
   FAIL  unpaired odd सुगम; discarded; restarting at है
   EVEN  है
Violations: 1 | pairs: 1 | borrows: 0 | even_feet: 2 | trailing_odd: 0 | word

PoemResult(score=0.85, lines=[LineResult(words=['चलो', 'हम', 'कहीं', 'दूर', 'चले'], score=1.0, violations=0, pairs=2, borrows=1, even_feet=1, trailing_odd=0, discarded_words=0, irregular_words=0, events=[Event(kind='pair', words=[0, 1, 2], text='चलो | हम+कहीं[0:1]', borrowed=True), Event(kind='even', words=[2], text='कहीं[1:2]', borrowed=False), Event(kind='pair', words=[3, 4], text='दूर | चले', borrowed=False)]), LineResult(words=['चलो', 'हम', 'और', 'कहीं', 'चले'], score=0.8, violations=1, pairs=1, borrows=0, even_feet=1, trailing_odd=1, discarded_words=1, irregular_words=0, events=[Event(kind='violation', words=[0], text='unpaired odd चलो; discarded; restarting at हम', borrowed=False), Event(kind='even', words=[1], text='हम', borrowed=False), Event(kind='pair', words=[2, 3], text='और | कहीं', borrowed=False), Event(kind='trailing_odd', words=[4], text='चले', borrowed=False)]), LineResult(words=['कार्यक्रम', 'बहुत', 'सुगम', 'है'], score=0.75, violations=1, pairs=1, borrows=0, even_fee

---
## Whole-Poem Input & Line-Wise Rhythm Evaluation

Paste an entire poem (even as a single paragraph) into `POEM` below using Devanagari full stops (`।` / `॥`) or newlines. The next cell splits it on the full-stop marks, prints the poem cleanly line-by-line, and evaluates every line and the overall poem using the same scanner and rhythm scorer.

In [6]:
# >>> PASTE WHOLE POEM HERE (uses । / ॥ full-stop marks or newlines) <<<
POEM = """माला फेरत जुग भया, फिरा न मन का फेर। कर का मनका डार दे, मन का मनका फेर॥"""

In [7]:
# Split Poem on Full-Stop Marks (। / ॥), Print Line-Wise, and Score
poem_lines = split_poem_lines(POEM)

print("FORMATTED POEM (line-wise):")
for idx, pline in enumerate(poem_lines, 1):
    print(f"  {idx}. {pline}")
print()

display_rhythm_report(poem_lines)

FORMATTED POEM (line-wise):
  1. माला फेरत जुग भया, फिरा न मन का फेर।
  2. कर का मनका डार दे, मन का मनका फेर॥

Line: माला फेरत जुग भया, फिरा न मन का फेर।
Words: माला=[2, 2](sum 4)  फेरत=[2, 1, 1](sum 4)  जुग=[1, 1](sum 2)  भया=[1, 2](sum 3)  फिरा=[1, 2](sum 3)  न=[1](sum 1)  मन=[1, 1](sum 2)  का=[2](sum 2)  फेर=[2, 1](sum 3)
Events:
   EVEN  माला
   EVEN  फेरत
   EVEN  जुग
   PAIR  भया | फिरा
   FAIL  unpaired odd न; discarded; restarting at मन
   EVEN  मन
   EVEN  का
   END   फेर
Violations: 1 | pairs: 1 | borrows: 0 | even_feet: 5 | trailing_odd: 1 | words: 9
LINE SCORE: 0.889
Line: कर का मनका डार दे, मन का मनका फेर॥
Words: कर=[1, 1](sum 2)  का=[2](sum 2)  मनका=[1, 1, 2](sum 4)  डार=[2, 1](sum 3)  दे=[2](sum 2)  मन=[1, 1](sum 2)  का=[2](sum 2)  मनका=[1, 1, 2](sum 4)  फेर=[2, 1](sum 3)
Events:
   EVEN  कर
   EVEN  का
   EVEN  मनका
   FAIL  unpaired odd डार; discarded; restarting at दे
   EVEN  दे
   EVEN  मन
   EVEN  का
   EVEN  मनका
   END   फेर
Violations: 1 | pairs: 0 | borrows: 0 

PoemResult(score=0.8888888888888888, lines=[LineResult(words=['माला', 'फेरत', 'जुग', 'भया', 'फिरा', 'न', 'मन', 'का', 'फेर'], score=0.8888888888888888, violations=1, pairs=1, borrows=0, even_feet=5, trailing_odd=1, discarded_words=1, irregular_words=1, events=[Event(kind='even', words=[0], text='माला', borrowed=False), Event(kind='even', words=[1], text='फेरत', borrowed=False), Event(kind='even', words=[2], text='जुग', borrowed=False), Event(kind='pair', words=[3, 4], text='भया | फिरा', borrowed=False), Event(kind='violation', words=[5], text='unpaired odd न; discarded; restarting at मन', borrowed=False), Event(kind='even', words=[6], text='मन', borrowed=False), Event(kind='even', words=[7], text='का', borrowed=False), Event(kind='trailing_odd', words=[8], text='फेर', borrowed=False)]), LineResult(words=['कर', 'का', 'मनका', 'डार', 'दे', 'मन', 'का', 'मनका', 'फेर'], score=0.8888888888888888, violations=1, pairs=0, borrows=0, even_feet=7, trailing_odd=1, discarded_words=1, irregular_words=